In [ ]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import cupy as cp
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, mean_squared_error, r2_score,
    precision_score, recall_score, f1_score, confusion_matrix, mean_absolute_error
)
from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression
from cuml.ensemble import RandomForestClassifier as cuRF, RandomForestRegressor as cuRFR
from cuml.linear_model import LogisticRegression as cuLogisticRegression, LinearRegression as cuLinearRegression
from cuml.svm import SVC as cuSVC, SVR as cuSVR
from cuml.model_selection import train_test_split as cu_train_test_split
from cuml.metrics import accuracy_score as cu_accuracy_score
from cuml import __version__ as cuml_version
import warnings
warnings.filterwarnings("ignore", message="Random state is currently ignored by probabilistic SVC")

try:
    from cuml.ensemble import GradientBoostingClassifier as cuGBC, GradientBoostingRegressor as cuGBR
    HAS_GRADIENT_BOOSTING = True
except ImportError:
    print("cuML Gradient Boosting 不可用，将跳过该模型")
    HAS_GRADIENT_BOOSTING = False

import xgboost as xgb
from datetime import datetime
from pandas.api.types import CategoricalDtype
import traceback
from sklearn.base import clone, BaseEstimator
from sklearn.model_selection import KFold
from typing import Dict, Any, Tuple, Optional, List
import warnings
import json
warnings.filterwarnings('ignore')
from docx import Document
from docx.shared import Inches
import copy
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import cm

os.environ['XGBOOST_DEVICE_MEMORY_LIMIT'] = '10GB'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

def clear_gpu_resources():
    """全面清理GPU资源并添加延迟"""
    gc.collect()
    time.sleep(0.5)
    
    if 'cp' in globals():
        try:
            cp._default_memory_pool.free_all_blocks()
            cp.get_default_memory_pool().free_all_blocks()
        except Exception: 
            pass
        
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass

def monitor_gpu_memory(threshold=0.8):
    """监控GPU内存使用，超过阈值则清理"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated()
        total = torch.cuda.get_device_properties(0).total_memory
        ratio = allocated / total
        if ratio > threshold:
            print(f"GPU内存使用率超过{threshold*100:.1f}% ({allocated/1e9:.2f}GB/{total/1e9:.2f}GB)，执行清理...")
            clear_gpu_resources()
            return True
    return False

def bootstrap_se_torch(X: np.ndarray, y: np.ndarray, n_bootstrap: int = 1000) -> Tuple[float, float, float]:
    """
    使用自助法计算标准误和置信区间（PyTorch优化版）
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_samples = len(y)
    
    bootstrap_effects = []
    
    for i in range(n_bootstrap):
        indices = np.random.choice(n_samples, n_samples, replace=True)
        X_boot = torch.tensor(X[indices], dtype=torch.float32, device=device)
        y_boot = torch.tensor(y[indices], dtype=torch.float32, device=device)
        
        with torch.no_grad():
            XtX = X_boot.t() @ X_boot
            Xty = X_boot.t() @ y_boot.reshape(-1, 1)
            
            XtX += torch.eye(X_boot.shape[1], device=device) * 1e-6
            
            coefficients = torch.linalg.solve(XtX, Xty)
            effect = coefficients[1].item() if coefficients.shape[0] > 1 else coefficients[0].item()
        
        bootstrap_effects.append(effect)
    
    bootstrap_effects = np.array(bootstrap_effects)
    se = np.std(bootstrap_effects, ddof=1)
    ci_lower = np.percentile(bootstrap_effects, 2.5)
    ci_upper = np.percentile(bootstrap_effects, 97.5)
    
    return se, ci_lower, ci_upper

def calculate_cluster_robust_se_torch(X: torch.Tensor, 
                                     y: torch.Tensor, 
                                     residuals: torch.Tensor, 
                                     cluster_ids: np.ndarray) -> float:
    """
    计算聚类稳健标准误（PyTorch版本）
    """
    device = X.device
    unique_clusters = np.unique(cluster_ids)
    n_clusters = len(unique_clusters)
    
    scores = []
    
    for cluster in unique_clusters:
        cluster_mask = (cluster_ids == cluster)
        X_cluster = X[cluster_mask]
        residuals_cluster = residuals[cluster_mask]
        
        cluster_score = X_cluster.t() @ residuals_cluster
        scores.append(cluster_score.cpu().numpy())
    
    scores_matrix = np.array(scores)
    mean_score = np.mean(scores_matrix, axis=0)
    cluster_variance = (scores_matrix - mean_score).T @ (scores_matrix - mean_score) / (n_clusters - 1)
    
    XtX_inv = torch.linalg.inv(X.t() @ X).cpu().numpy()
    cluster_vcov = XtX_inv @ cluster_variance @ XtX_inv
    se = np.sqrt(np.diag(cluster_vcov))[1]
    
    return se

class FirstStageModelFactory:
    """第一阶段模型工厂类，用于创建和配置不同的机器学习模型"""
    def __init__(self):
        self.model_configs = {
            'RandomForest': {
                'classification': lambda: cuRF(n_estimators=200, max_depth=20, random_state=42),
                'regression': lambda: cuRFR(n_estimators=200, max_depth=20, random_state=42)
            },
            'LogisticRegression': {
                'classification': lambda: cuLogisticRegression(penalty='l2', C=1.0, max_iter=1000),
                'regression': lambda: cuLinearRegression(fit_intercept=True, algorithm='eig')
            },
            'GradientBoosting': {
                'classification': lambda: cuGBC(n_estimators=200, max_depth=6, random_state=42) if HAS_GRADIENT_BOOSTING else None,
                'regression': lambda: cuGBR(n_estimators=200, max_depth=6, random_state=42) if HAS_GRADIENT_BOOSTING else None
            },
            'SVC': {
                'classification': lambda: cuSVC(kernel='rbf', C=1.0, probability=True, random_state=42),
                'regression': lambda: cuSVR(kernel='rbf', C=1.0)
            },
            'XGBoost': {
                'classification': lambda: xgb.XGBClassifier(n_estimators=200, max_depth=6, random_state=42, tree_method='gpu_hist'),
                'regression': lambda: xgb.XGBRegressor(n_estimators=200, max_depth=6, random_state=42, tree_method='gpu_hist')
            },
            'MLP': {
                'classification': lambda input_size: PyTorchMLP(input_size=input_size, output_size=1, task_type='classification'),
                'regression': lambda input_size: PyTorchMLP(input_size=input_size, output_size=1, task_type='regression')
            }
        }
    
    def get_model(self, model_name, task_type, input_size=None):
        """
        根据模型名称和任务类型获取模型实例
        """
        if model_name not in self.model_configs:
            raise ValueError(f"不支持的模型类型: {model_name}")
        
        if task_type not in self.model_configs[model_name]:
            raise ValueError(f"模型 {model_name} 不支持任务类型: {task_type}")
        
        if model_name == 'MLP':
            if input_size is None:
                raise ValueError("MLP模型需要指定input_size参数")
            return self.model_configs[model_name][task_type](input_size)
        else:
            model = self.model_configs[model_name][task_type]()
            if model is None:
                raise ValueError(f"模型 {model_name} 不可用，可能缺少依赖")
            return model

class PyTorchMLP(nn.Module):
    """PyTorch多层感知机模型（GPU加速版），用于替代cuML的MLP"""
    def __init__(self, input_size, hidden_sizes=[100, 50], output_size=1, 
                 task_type='classification', dropout_rate=0.1):
        super(PyTorchMLP, self).__init__()
        self.task_type = task_type
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, output_size))
        
        if task_type == 'classification':
            layers.append(nn.Sigmoid())
        
        self.network = nn.Sequential(*layers).to(self.device)
        
    def forward(self, x):
        return self.network(x)
    
    def fit(self, X, y, epochs=100, batch_size=256, lr=0.001):
        """训练模型"""
        self.train()
        
        if not isinstance(X, torch.Tensor):
            X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        else:
            X_tensor = X.to(self.device)
            
        if not isinstance(y, torch.Tensor):
            y_tensor = torch.tensor(y, dtype=torch.float32, device=self.device)
        else:
            y_tensor = y.to(self.device)
        
        if self.task_type == 'classification':
            y_tensor = y_tensor.view(-1, 1)
        
        if self.task_type == 'classification':
            criterion = nn.BCELoss()
        else:
            criterion = nn.MSELoss()
        
        optimizer = optim.Adam(self.parameters(), lr=lr)
        
        dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
        
        for epoch in range(epochs):
            epoch_loss = 0.0
            for batch_X, batch_y in dataloader:
                optimizer.zero_grad()
                outputs = self(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            if (epoch + 1) % 20 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(dataloader):.4f}")
    
    def predict(self, X):
        """预测"""
        self.eval()
        with torch.no_grad():
            if not isinstance(X, torch.Tensor):
                X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            else:
                X_tensor = X.to(self.device)
                
            predictions = self(X_tensor)
            
            if self.task_type == 'classification':
                return (predictions > 0.5).float().cpu().numpy().flatten()
            else:
                return predictions.cpu().numpy().flatten()
    
    def predict_proba(self, X):
        """预测概率（仅分类任务）"""
        if self.task_type != 'classification':
            raise ValueError("predict_proba仅适用于分类任务")
        
        self.eval()
        with torch.no_grad():
            if not isinstance(X, torch.Tensor):
                X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            else:
                X_tensor = X.to(self.device)
                
            return self(X_tensor).cpu().numpy()

class CustomVotingClassifier(BaseEstimator):
    """自定义投票分类器，支持cuML和PyTorch模型的软投票集成"""
    def __init__(self, estimators, weights=None):
        self.estimators = estimators  # list of (name, unfitted_model)
        if weights is None:
            self.weights = [1.0 / len(estimators)] * len(estimators)
        else:
            # Convert weights to list if they are a NumPy array
            self.weights = weights.tolist() if isinstance(weights, np.ndarray) else weights
            # Normalize weights to sum to 1
            weights_sum = sum(self.weights)
            if weights_sum > 0:
                self.weights = [w / weights_sum for w in self.weights]
            else:
                self.weights = [1.0 / len(estimators)] * len(estimators)
    
    def fit(self, X, y):
        for _, model in self.estimators:
            if isinstance(X, cp.ndarray):
                X_fit = X
                y_fit = y if isinstance(y, cp.ndarray) else cp.array(y)
            elif isinstance(X, np.ndarray):
                X_fit = cp.array(X)
                y_fit = cp.array(y)
            else:
                X_fit = X
                y_fit = y
            
            model.fit(X_fit, y_fit)
        return self
    
    def predict_proba(self, X):
        # 统一输入为 cp.ndarray
        if isinstance(X, np.ndarray):
            X = cp.array(X)
        elif not isinstance(X, cp.ndarray):
            X = cp.array(X)
        
        probas = []
        for i, (_, model) in enumerate(self.estimators):
            # 获取 proba，并统一转换为 cp.ndarray
            if hasattr(model, 'predict_proba'):
                proba = model.predict_proba(X)
            else:
                pred = model.predict(X)
                proba = cp.zeros((len(X), 2))
                proba[:, 0] = 1 - pred
                proba[:, 1] = pred
            
            # 统一转换为 cp.ndarray（如果已经是 cp，无影响；如果是 np，会复制到 GPU）
            proba = cp.array(proba)
            
            if proba.ndim == 1:
                proba = cp.stack([1 - proba, proba], axis=1)
            
            # 加权并 append
            probas.append(proba * self.weights[i])
        
        # sum(probas) 现在所有元素都是 cp.ndarray，可以安全相加
        total_weight = sum(self.weights)
        if total_weight == 0:
            total_weight = 1.0  # 避免除零
        avg_proba = sum(probas) / total_weight
        
        # 返回时转换为 np.ndarray，以兼容后续 NumPy 操作（如残差计算）
        return cp.asnumpy(avg_proba)
    
    def predict(self, X):
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)  # 直接用 np.argmax，因为 proba 已转换为 np

def get_y_residuals(model_proto, X: np.ndarray, y: np.ndarray, n_splits=5):
    """
    使用五折交叉验证计算outcome (y) 的残差
    对于y，使用predict_proba[:,1] 计算残差（修复为使用概率）
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    residuals = np.zeros(len(y))
    
    for train_idx, test_idx in kf.split(X):
        model_clone = clone(model_proto) if not isinstance(model_proto, CustomVotingClassifier) else copy.deepcopy(model_proto)
        
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        model_clone.fit(X_train, y_train)
        
        if hasattr(model_clone, 'predict_proba'):
            y_pred = model_clone.predict_proba(X_test)[:, 1] if model_clone.predict_proba(X_test).ndim > 1 else model_clone.predict_proba(X_test)
        else:
            y_pred = model_clone.predict(X_test)
        
        residuals[test_idx] = y_test - y_pred
    
    return residuals

def get_t_residuals(model_proto, X: np.ndarray, t: np.ndarray, n_splits=5):
    """
    使用五折交叉验证计算treatment (t) 的残差
    对于t，使用predict_proba[:,1]
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    residuals = np.zeros(len(t))
    
    for train_idx, test_idx in kf.split(X):
        model_clone = clone(model_proto) if not isinstance(model_proto, CustomVotingClassifier) else copy.deepcopy(model_proto)
        
        X_train, X_test = X[train_idx], X[test_idx]
        t_train, t_test = t[train_idx], t[test_idx]
        
        model_clone.fit(X_train, t_train)
        
        if hasattr(model_clone, 'predict_proba'):
            t_pred = model_clone.predict_proba(X_test)[:, 1] if model_clone.predict_proba(X_test).ndim > 1 else model_clone.predict_proba(X_test)
        else:
            t_pred = model_clone.predict(X_test)
        
        residuals[test_idx] = t_test - t_pred
    
    return residuals

from scipy import stats
def second_stage_dml_analysis(t_residuals, y_residuals, X_second=None, robust_method='bootstrap', n_bootstrap=1000):
    """
    执行DML的第二阶段分析，计算处理效应及统计检验指标。
    """
    if torch.is_tensor(t_residuals):
        t_residuals = t_residuals.cpu().numpy() if t_residuals.is_cuda else t_residuals.numpy()
    if torch.is_tensor(y_residuals):
        y_residuals = y_residuals.cpu().numpy() if y_residuals.is_cuda else y_residuals.numpy()
    if X_second is not None and torch.is_tensor(X_second):
        X_second = X_second.cpu().numpy() if X_second.is_cuda else X_second.numpy()
    
    n = len(y_residuals)
    
    t_residuals = np.asarray(t_residuals).reshape(-1, 1)
    y_residuals = np.asarray(y_residuals).reshape(-1, 1)
    
    if X_second is not None:
        X_second = np.asarray(X_second)
        interaction_terms = t_residuals * X_second
        design_matrix = np.column_stack((t_residuals, interaction_terms))
    else:
        design_matrix = t_residuals
    
    coefficients, _, _, _ = np.linalg.lstsq(design_matrix, y_residuals, rcond=None)
    
    effect = coefficients[0][0] if len(coefficients) > 0 else 0.0
    
    y_pred = np.dot(design_matrix, coefficients)
    residuals = y_residuals - y_pred
    
    se = 0.0
    t_statistic = 0.0
    p_value = 1.0
    f_statistic = 0.0
    f_p_value = 1.0
    
    if robust_method == 'bootstrap':
        bootstrap_effects = []
        for _ in range(n_bootstrap):
            indices = np.random.choice(n, n, replace=True)
            t_boot = t_residuals[indices]
            y_boot = y_residuals[indices]
            if X_second is not None:
                X_second_boot = X_second[indices]
                interaction_boot = t_boot * X_second_boot
                design_boot = np.column_stack((t_boot, interaction_boot))
            else:
                design_boot = t_boot
            coef_boot, _, _, _ = np.linalg.lstsq(design_boot, y_boot, rcond=None)
            bootstrap_effects.append(coef_boot[0][0] if len(coef_boot) > 0 else 0.0)
        se = np.std(bootstrap_effects, ddof=1) if bootstrap_effects else 0.0
    else:
        mse = np.sum(residuals**2) / (n - design_matrix.shape[1])
        xtx_inv = np.linalg.inv(np.dot(design_matrix.T, design_matrix))
        se_matrix = mse * xtx_inv
        se = np.sqrt(np.diag(se_matrix)[0]) if se_matrix.size > 0 else 0.0
    
    ci_lower = effect - 1.96 * se if se > 0 else effect
    ci_upper = effect + 1.96 * se if se > 0 else effect
    
    if se > 0:
        t_statistic = effect / se
        df = n - design_matrix.shape[1]
        p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), df))
    
    ss_total = np.sum((y_residuals - np.mean(y_residuals))**2)
    ss_residual = np.sum(residuals**2)
    if ss_residual > 0 and design_matrix.shape[1] > 0:
        ss_explained = ss_total - ss_residual
        df_model = design_matrix.shape[1]
        df_resid = n - df_model
        f_statistic = (ss_explained / df_model) / (ss_residual / df_resid) if df_resid > 0 else 0.0
        f_p_value = 1 - stats.f.cdf(f_statistic, df_model, df_resid) if df_resid > 0 else 1.0
    
    return {
        'effect': effect,
        'se': se,
        'lb': ci_lower,
        'ub': ci_upper,
        't_statistic': t_statistic,
        'p_value': p_value,
        'f_statistic': f_statistic,
        'f_p_value': f_p_value,
        'df': n - design_matrix.shape[1] if design_matrix.shape[1] > 0 else n
    }

def torch_dml_with_pretrained(X: np.ndarray, 
                              y: np.ndarray, 
                              t: np.ndarray, 
                              model_y: BaseEstimator, 
                              model_t: BaseEstimator, 
                              use_robust_se: bool = True, 
                              robust_method: str = 'bootstrap',
                              n_bootstrap: int = 1000,
                              cluster_ids: Optional[np.ndarray] = None,
                              n_splits: int = 5) -> Dict[str, Any]:
    """
    使用预训练的第一阶段模型执行DML的第二阶段分析（残差回归）
    """
    print("执行DML第二阶段分析（残差回归）...")
    
    effect = 0.0
    se = 0.0
    ci_lower = 0.0
    ci_upper = 0.0
    t_statistic = 0.0
    p_value = 1.0
    f_statistic = 0.0
    f_p_value = 1.0
    df = 0
    
    X_np = X if isinstance(X, np.ndarray) else cp.asnumpy(X)
    y_np = y if isinstance(y, np.ndarray) else cp.asnumpy(y)
    t_np = t if isinstance(t, np.ndarray) else cp.asnumpy(t)
    
    print("使用交叉验证计算第一阶段残差...")
    y_residuals = get_y_residuals(model_y, X_np, y_np, n_splits=n_splits)
    t_residuals = get_t_residuals(model_t, X_np, t_np, n_splits=n_splits)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    t_residuals_tensor = torch.tensor(t_residuals, dtype=torch.float32, device=device).reshape(-1, 1)
    y_residuals_tensor = torch.tensor(y_residuals, dtype=torch.float32, device=device)
    
    ones = torch.ones_like(t_residuals_tensor, device=device)
    X_second = torch.cat([ones, t_residuals_tensor], dim=1)
    
    with torch.no_grad():
        XtX = X_second.t() @ X_second
        Xty = X_second.t() @ y_residuals_tensor.reshape(-1, 1)
        
        XtX += torch.eye(2, device=device) * 1e-6
        
        coefficients = torch.linalg.solve(XtX, Xty)
        effect = coefficients[1].item()
    
    if use_robust_se and robust_method == 'bootstrap':
        print("使用自助法计算稳健标准误...")
        se, ci_lower, ci_upper = bootstrap_se_torch(
            X_second.cpu().numpy(), 
            y_residuals_tensor.cpu().numpy(), 
            n_bootstrap=n_bootstrap
        )
        
        if se > 0:
            t_statistic = effect / se
            n, k = X_second.shape
            df = n - k
            p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), df))
    else:
        n, k = X_second.shape
        with torch.no_grad():
            y_pred_tensor = X_second @ coefficients
            residuals = y_residuals_tensor.reshape(-1, 1) - y_pred_tensor
        mse = torch.sum(residuals**2) / (n - k)
        cov_matrix = torch.linalg.inv(XtX) * mse
        se = torch.sqrt(torch.diag(cov_matrix))[1].item()
        
        t_stat_val = 1.96
        ci_lower = effect - t_stat_val * se
        ci_upper = effect + t_stat_val * se
        
        t_statistic = effect / se
        df = n - k
        p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), df))
        
        with torch.no_grad():
            y_pred_tensor = X_second @ coefficients
            ss_residual = torch.sum((y_residuals_tensor.reshape(-1, 1) - y_pred_tensor)**2)
            ss_total = torch.sum((y_residuals_tensor.reshape(-1, 1) - torch.mean(y_residuals_tensor))**2)
            ss_explained = ss_total - ss_residual
    
            f_statistic = (ss_explained / (k - 1)) / (ss_residual / (n - k))
            f_p_value = 1 - stats.f.cdf(f_statistic, k - 1, n - k)
    
    print(f"DML处理效应估计: {effect:.6f}")
    print(f"标准误: {se:.6f}")
    print(f"95%置信区间: [{ci_lower:.6f}, {ci_upper:.6f}]")
    print(f"t统计量: {t_statistic:.4f}")
    print(f"p值: {p_value:.6f}")
    print(f"F统计量: {f_statistic:.4f}")
    print(f"F检验p值: {f_p_value:.6f}")
    
    return {
        'effect': effect,
        'se': se,
        'lb': ci_lower,
        'ub': ci_upper,
        't_statistic': t_statistic,
        'p_value': p_value,
        'f_statistic': f_statistic,
        'f_p_value': f_p_value,
        'df': df
    }

def first_stage_cuml_enhanced(X, y, task_type='regression', model_name='RandomForest', test_size=0.25):
    """
    增强版第一阶段预测，支持多种模型
    """
    clear_gpu_resources()
    
    X_train, X_test, y_train, y_test = cu_train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    
    model_factory = FirstStageModelFactory()
    
    try:
        if model_name == 'MLP':
            input_size = X_train.shape[1]
            model = model_factory.get_model(model_name, task_type, input_size)
        else:
            model = model_factory.get_model(model_name, task_type)
        
        if model_name == 'MLP':
            X_train_pt = torch.tensor(cp.asnumpy(X_train), dtype=torch.float32)
            y_train_pt = torch.tensor(cp.asnumpy(y_train), dtype=torch.float32)
            
            model.fit(X_train_pt, y_train_pt)
            
            X_test_pt = torch.tensor(cp.asnumpy(X_test), dtype=torch.float32)
            y_pred = model.predict(X_test_pt)
            y_pred = cp.array(y_pred)
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
        
        metrics = evaluate_model(model, X_test, y_test, model_name, task_type)
        
        metrics['model_name'] = model_name
        metrics['task_type'] = task_type
        
        if task_type == 'classification':
            summary = f"准确率: {metrics.get('accuracy', 0):.4f}, F1: {metrics.get('f1', 0):.4f}, AUC: {metrics.get('auc', 0):.4f}"
        else:
            summary = f"R²: {metrics.get('r2', 0):.4f}, RMSE: {metrics.get('rmse', 0):.4f}, MAE: {metrics.get('mae', 0):.4f}"
        
        print(f"{model_name} {task_type}模型评估 - {summary}")
        
        return model, metrics
        
    except Exception as e:
        print(f"训练{model_name}模型失败: {str(e)}")
        traceback.print_exc()
        return None, None

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
def evaluate_model(model, X_test, y_test, model_name="Model", task_type='classification'):
    """
    评估模型性能，根据任务类型使用不同的指标
    """
    try:
        if hasattr(X_test, 'get'):
            X_test_np = X_test.get()
        elif hasattr(X_test, 'cpu'):
            X_test_np = X_test.cpu().numpy()
        elif hasattr(X_test, 'numpy'):
            X_test_np = X_test.numpy()
        else:
            X_test_np = np.array(X_test)
            
        if hasattr(y_test, 'get'):
            y_test_np = y_test.get()
        elif hasattr(y_test, 'cpu'):
            y_test_np = y_test.cpu().numpy()
        elif hasattr(y_test, 'numpy'):
            y_test_np = y_test.numpy()
        else:
            y_test_np = np.array(y_test)
        
        y_pred = model.predict(X_test_np)
        
        if hasattr(y_pred, 'get'):
            y_pred_np = y_pred.get()
        elif hasattr(y_pred, 'cpu'):
            y_pred_np = y_pred.cpu().numpy()
        elif hasattr(y_pred, 'numpy'):
            y_pred_np = y_pred.numpy()
        else:
            y_pred_np = np.array(y_pred)
        
        metrics = {}
        
        if task_type == 'regression':
            metrics['r2'] = r2_score(y_test_np, y_pred_np)
            metrics['mse'] = mean_squared_error(y_test_np, y_pred_np)
            metrics['rmse'] = np.sqrt(metrics['mse'])
            metrics['mae'] = mean_absolute_error(y_test_np, y_pred_np)
            
            print(f"{model_name} 回归模型评估 - R²: {metrics['r2']:.4f}, RMSE: {metrics['rmse']:.4f}, MAE: {metrics['mae']:.4f}")
            
        else:
            metrics['accuracy'] = accuracy_score(y_test_np, y_pred_np)
            metrics['precision'] = precision_score(y_test_np, y_pred_np, average='weighted')
            metrics['recall'] = recall_score(y_test_np, y_pred_np, average='weighted')
            metrics['f1'] = f1_score(y_test_np, y_pred_np, average='weighted')
            
            auc_score = np.nan
            try:
                if hasattr(model, 'predict_proba'):
                    y_proba = model.predict_proba(X_test_np)
                    if hasattr(y_proba, 'get'):
                        y_proba = y_proba.get()
                    elif hasattr(y_proba, 'cpu'):
                        y_proba = y_proba.cpu().numpy()
                    
                    if len(y_proba.shape) > 1 and y_proba.shape[1] == 2:
                        auc_score = roc_auc_score(y_test_np, y_proba[:, 1])
                    elif len(y_proba.shape) > 1:
                        auc_score = roc_auc_score(y_test_np, y_proba, multi_class='ovr')
                    else:
                        auc_score = roc_auc_score(y_test_np, y_proba)
                elif hasattr(model, 'decision_function'):
                    y_score = model.decision_function(X_test_np)
                    if hasattr(y_score, 'get'):
                        y_score = y_score.get()
                    elif hasattr(y_score, 'cpu'):
                        y_score = y_score.cpu().numpy()
                    auc_score = roc_auc_score(y_test_np, y_score)
                else:
                    print(f"警告: 模型 {model_name} 不支持概率预测，无法计算AUC")
                    auc_score = np.nan
            except Exception as e:
                print(f"警告: 计算模型 {model_name} 的AUC时出错: {str(e)}")
                auc_score = np.nan
            
            metrics['auc'] = auc_score
            print(f"{model_name} 分类模型评估 - 准确率: {metrics['accuracy']:.4f}, 精确率: {metrics['precision']:.4f}, 召回率: {metrics['recall']:.4f}, F1: {metrics['f1']:.4f}, AUC: {metrics['auc']:.4f}")
        
        metrics['predictions'] = y_pred_np
        return metrics
        
    except Exception as e:
        print(f"评估模型 {model_name} 时出错: {str(e)}")
        traceback.print_exc()
        if task_type == 'regression':
            return {
                'r2': np.nan,
                'mse': np.nan,
                'rmse': np.nan,
                'mae': np.nan,
                'predictions': None
            }
        else:
            return {
                'accuracy': np.nan,
                'precision': np.nan,
                'recall': np.nan,
                'f1': np.nan,
                'auc': np.nan,
                'predictions': None
            }

class ModelComparator:
    """模型比较器，用于比较不同模型的性能并选择最佳模型"""
    
    def __init__(self, task_type):
        self.task_type = task_type
        self.model_metrics = {}
        self.weights = self._get_default_weights()
    
    def add_model(self, model_name, metrics):
        """添加模型及其评估指标"""
        self.model_metrics[model_name] = metrics
    
    def _get_default_weights(self):
        """获取默认指标权重"""
        if self.task_type == 'classification':
            return {'accuracy': 0.2, 'precision': 0.2, 'recall': 0.2, 'f1': 0.2, 'auc': 0.2}
        else:
            return {'r2': 0.4, 'rmse': 0.3, 'mae': 0.3}
    
    def _calculate_score(self, metrics):
        """计算模型综合得分（修复版）"""
        if self.task_type == 'regression':
            r2 = metrics.get('r2', 0)
            rmse = metrics.get('rmse', 1e6)
            mae = metrics.get('mae', 1e6)
            if r2 < 0:
                return -1.0 * abs(r2)
            score = r2 * 0.7 + (1 / (rmse + 1e-6)) * 0.15 + (1 / (mae + 1e-6)) * 0.15
            return score
        else:
            accuracy = metrics.get('accuracy', 0)
            precision = metrics.get('precision', 0)
            recall = metrics.get('recall', 0)
            f1 = metrics.get('f1', 0)
            auc = metrics.get('auc', 0)
            if np.isnan(auc):
                auc = 0
            if auc > 0:
                score = accuracy * 0.2 + precision * 0.2 + recall * 0.2 + f1 * 0.2 + auc * 0.2
            else:
                score = accuracy * 0.25 + precision * 0.25 + recall * 0.25 + f1 * 0.25
            return score
    
    def compare_models(self):
        """比较所有模型并返回排名"""
        model_scores = {}
        
        for model_name, metrics in self.model_metrics.items():
            score = self._calculate_score(metrics)
            model_scores[model_name] = score
        
        sorted_models = sorted(model_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_models
    
    def get_best_model(self):
        """获取最佳模型名称"""
        sorted_models = self.compare_models()
        return sorted_models[0][0] if sorted_models else None
    
    def print_comparison(self):
        """打印模型比较结果"""
        print(f"\n=== {self.task_type.upper()} 模型比较结果 ===")
        sorted_models = self.compare_models()
        
        for rank, (model_name, score) in enumerate(sorted_models, 1):
            metrics = self.model_metrics[model_name]
            
            if self.task_type == 'classification':
                summary = f"准确率: {metrics.get('accuracy', 0):.4f}, 精确率: {metrics.get('precision', 0):.4f}, 召回率: {metrics.get('recall', 0):.4f}, F1: {metrics.get('f1', 0):.4f}, AUC: {metrics.get('auc', 0):.4f}"
            else:
                summary = f"R²: {metrics.get('r2', 0):.4f}, RMSE: {metrics.get('rmse', 0):.4f}, MAE: {metrics.get('mae', 0):.4f}"
            
            print(f"{rank}. {model_name}: 得分={score:.4f}, {summary}")

class SensitivityAnalysisOptimized:
    """敏感性分析类（优化版）"""
    
    def __init__(self, X: np.ndarray, y: np.ndarray, t: np.ndarray, control_variables: list):
        self.X = X
        self.y = y
        self.t = t
        self.control_variables = control_variables
        self.results = {}

    def run_analysis(self, variations: Dict[str, Dict[str, Any]], n_splits: int = 5) -> Dict[str, Any]:
        """运行敏感性分析"""
        for name, variation in variations.items():
            print(f"运行敏感性分析: {name}")
            
            try:
                X_var = self.X
                if 'control_vars' in variation:
                    control_indices = [i for i, var in enumerate(self.control_variables) 
                                     if var in variation['control_vars']]
                    X_var = self.X[:, control_indices]
                
                print("训练第一阶段模型...")
                
                model_y, metrics_y = first_stage_cuml_enhanced(
                    cp.array(X_var), cp.array(self.y), 
                    task_type='classification', 
                    model_name=variation.get('model_name', 'RandomForest')
                )
                model_t, metrics_t = first_stage_cuml_enhanced(
                    cp.array(X_var), cp.array(self.t), 
                    task_type='classification', 
                    model_name=variation.get('model_name', 'RandomForest')
                )
                
                y_residuals = get_y_residuals(model_y, X_var, self.y, n_splits=n_splits)
                t_residuals = get_t_residuals(model_t, X_var, self.t, n_splits=n_splits)
                
                dml_result = second_stage_dml_analysis(
                    t_residuals=t_residuals,
                    y_residuals=y_residuals,
                    robust_method=variation.get('robust_method', 'bootstrap'),
                    n_bootstrap=variation.get('n_bootstrap', 1000)
                )
                
                effect = dml_result['effect']
                se = dml_result['se']
                lb = dml_result['lb']
                ub = dml_result['ub']
                t_statistic = dml_result.get('t_statistic', np.nan)
                p_value = dml_result.get('p_value', np.nan)
                f_statistic = dml_result.get('f_statistic', np.nan)
                f_p_value = dml_result.get('f_p_value', np.nan)
                
                self.results[name] = {
                    'effect': effect,
                    'se': se,
                    'lb': lb,
                    'ub': ub,
                    't_statistic': t_statistic,
                    'p_value': p_value,
                    'f_statistic': f_statistic,
                    'f_p_value': f_p_value,
                    'metrics_y': metrics_y,
                    'metrics_t': metrics_t,
                    'variation': variation
                }
                
                print(f"{name}: 效应={effect:.6f}, 标准误={se:.6f}")
                
            except Exception as e:
                print(f"敏感性分析 {name} 出错: {str(e)}")
                traceback.print_exc()
                self.results[name] = {'error': str(e)}
        
        return self.results
    
    def summarize(self):
        """
        总结并格式化输出敏感性分析结果
        """
        summary_lines = []
        summary_lines.append("\n=== 敏感性分析结果总结 ===")
        summary_lines.append("场景\t\t处理效应\t标准误\t95%置信区间")
        summary_lines.append("-" * 65)
        
        for name, result in self.results.items():
            if 'error' not in result:
                effect = result['effect']
                se = result['se']
                lb = result['lb']
                ub = result['ub']
                summary_lines.append(f"{name[:12]:<12}\t{effect:.6f}\t{se:.6f}\t[{lb:.6f}, {ub:.6f}]")
            else:
                summary_lines.append(f"{name[:12]:<12}\t分析出错: {result['error']}")
        
        successful_results = [res for res in self.results.values() if 'error' not in res]
        if len(successful_results) > 0:
            effects = [res['effect'] for res in successful_results]
            min_effect = min(effects)
            max_effect = max(effects)
            range_effect = max_effect - min_effect
            summary_lines.append("-" * 65)
            summary_lines.append(f"处理效应范围: {min_effect:.6f} 到 {max_effect:.6f} (范围: {range_effect:.6f})")
            summary_lines.append(f"效应变化幅度: {(range_effect / np.mean(effects)) * 100:.2f}%")
        
        return "\n".join(summary_lines)

class DataPreprocessor:
    """数据预处理类，封装数据读取和预处理逻辑"""
    
    def __init__(self, control_variables):
        self.control_variables = control_variables
        self.scalers = {}
        self.imputers = {}
        self.encoders = {}
    
    def load_and_preprocess_data(self, file_path):
        """加载并预处理数据"""
        print("步骤1: 数据准备")
        print(f"使用文件: {file_path}")
        
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"文件 '{file_path}' 不存在")
        
        data = pd.read_stata(file_path)
        
        y_series = data['entrepreneurship']
        t_series = data['mobile_use']
        
        le = LabelEncoder()
        y = le.fit_transform(y_series) if y_series.dtype == 'object' or isinstance(y_series.dtype, CategoricalDtype) else y_series.values
        t = le.fit_transform(t_series) if t_series.dtype == 'object' or isinstance(t_series.dtype, CategoricalDtype) else t_series.values

        missing_cols = [col for col in self.control_variables if col not in data.columns]
        if missing_cols:
            raise ValueError(f"数据中缺少以下控制变量: {missing_cols}")
        
        for col in self.control_variables:
            if data[col].dtype == 'object' or isinstance(data[col].dtype, CategoricalDtype):
                self.encoders[col] = LabelEncoder()
                data[col] = self.encoders[col].fit_transform(data[col].astype(str))
            
            if data[col].isnull().sum() > 0:
                self.imputers[col] = SimpleImputer(strategy='mean')
                data[col] = self.imputers[col].fit_transform(data[col].values.reshape(-1, 1)).flatten()
            
            self.scalers[col] = StandardScaler()
            data[col] = self.scalers[col].fit_transform(data[col].values.reshape(-1, 1)).flatten()

        X = data[self.control_variables].values
        
        X_gpu = cp.array(X, dtype=cp.float32)
        y_gpu = cp.array(y, dtype=cp.float32)
        t_gpu = cp.array(t, dtype=cp.float32)
        
        print("检查数据完整性：")
        print("X 中是否有 NaN 或 Inf:", cp.any(cp.isnan(X_gpu)), cp.any(cp.isinf(X_gpu)))
        print("y 中是否有 NaN 或 Inf:", cp.any(cp.isnan(y_gpu)), cp.any(cp.isinf(y_gpu)))
        print("t 中是否有 NaN 或 Inf:", cp.any(cp.isnan(t_gpu)), cp.any(cp.isinf(t_gpu)))
        
        X_gpu = cp.nan_to_num(X_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        y_gpu = cp.nan_to_num(y_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        t_gpu = cp.nan_to_num(t_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        
        print(f"数据形状: X={X_gpu.shape}, y={y_gpu.shape}, t={t_gpu.shape}")
        
        return cp.asnumpy(X_gpu), cp.asnumpy(y_gpu), cp.asnumpy(t_gpu), data

def plot_ensemble_weights(model_names, weights, save_path, filename, title):
    """
    绘制集成模型中各模型权重的横向条形图，使用viridis颜色方案。
    调整x轴范围以放大微小差异。
    """
    df = pd.DataFrame({
        'Model': model_names,
        'Weight': weights
    })
    # 按权重降序排序（高权重在上）
    df_sorted = df.sort_values(by='Weight', ascending=False)
    
    sns.set_context("notebook", rc={"axes.titlesize": 13, "axes.labelsize": 13, "xtick.labelsize": 13, "ytick.labelsize": 13})
    plt.figure(figsize=(12, 10))
    
    # 使用viridis调色板
    n_colors = len(df_sorted)
    palette = cm.viridis(np.linspace(0, 1, n_colors))
    
    sns.barplot(x="Weight", y="Model", data=df_sorted, palette=palette, orient="h")
    plt.xlabel("Ensemble Weight", fontsize=13)
    plt.ylabel(" ")
    plt.title(title, fontsize=14)
    
    # 创建图例
    legend_labels = [mpatches.Patch(color=palette[i], label=df_sorted['Model'].iloc[i]) for i in range(len(df_sorted))]
    plt.legend(handles=legend_labels, title="Model", bbox_to_anchor=(1.05, 0.8), loc='upper left', ncol=2, fontsize=13, title_fontsize=14, frameon=False)
    
    weights = df_sorted['Weight'].values
    min_w = min(weights)
    max_w = max(weights)
    delta = (max_w - min_w) * 0.05 if max_w > min_w else 0.01
    plt.xlim(min_w - delta, max_w + delta)
    
    plt.grid(True, linestyle='--', linewidth=0.7, alpha=0.7)
    plt.tight_layout()
    
    full_path = os.path.join(save_path, f"{filename}.pdf")
    plt.savefig(full_path, format='pdf', bbox_inches='tight', dpi=1200)
    plt.close()
    print(f"集成模型权重图已保存到: {full_path}")

def save_detailed_results(effect, se, lb, ub, t_statistic, p_value, f_statistic, f_p_value, model_y, model_t, metrics_y, metrics_t,
                         sensitivity_results, save_path, robust_method='bootstrap',
                         batch_id="batch_001"):
    """
    保存详细的结果到Word文件
    """
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    doc = Document()
    doc.add_heading(f'分析结果 - {batch_id}', 0)
    
    doc.add_heading('主要分析结果', level=1)
    para = doc.add_paragraph()
    para.add_run(f"处理效应估计: {effect:.6f}\n")
    para.add_run(f"标准误差: {se:.6f}\n")
    para.add_run(f"置信区间下界: {lb:.6f}\n")
    para.add_run(f"置信区间上界: {ub:.6f}\n")
    para.add_run(f"p值: {p_value:.6f}\n")
    para.add_run(f"t检验值: {t_statistic:.4f}\n")
    para.add_run(f"F检验值: {f_statistic:.4f}\n")
    para.add_run(f"f检验P值: {f_p_value:.6f}\n")
    para.add_run(f"Y模型类型: {type(model_y).__name__}\n")
    para.add_run(f"T模型类型: {type(model_t).__name__}\n")
    para.add_run(f"稳健方法: {robust_method}")
    
    doc.add_heading('Y模型指标', level=2)
    para = doc.add_paragraph()
    para.add_run(f"准确率: {metrics_y.get('accuracy', np.nan):.4f}\n")
    para.add_run(f"F1: {metrics_y.get('f1', np.nan):.4f}\n")
    para.add_run(f"AUC: {metrics_y.get('auc', np.nan):.4f}")
    
    doc.add_heading('T模型指标', level=2)
    para = doc.add_paragraph()
    para.add_run(f"准确率: {metrics_t.get('accuracy', np.nan):.4f}\n")
    para.add_run(f"精确率: {metrics_t.get('precision', np.nan):.4f}\n")
    para.add_run(f"召回率: {metrics_t.get('recall', np.nan):.4f}\n")
    para.add_run(f"F1: {metrics_t.get('f1', np.nan):.4f}\n")
    para.add_run(f"AUC: {metrics_t.get('auc', np.nan):.4f}")
    
    doc.add_heading('敏感性分析结果', level=1)
    for name, result in sensitivity_results.items():
        if 'error' not in result:
            doc.add_heading(f'场景: {name}', level=2)
            para = doc.add_paragraph()
            para.add_run(f"处理效应: {result.get('effect', np.nan):.6f}\n")
            para.add_run(f"标准误差: {result.get('se', np.nan):.6f}\n")
            para.add_run(f"置信区间下界: {result.get('lb', np.nan):.6f}\n")
            para.add_run(f"置信区间上界: {result.get('ub', np.nan):.6f}\n")
            para.add_run(f"t统计量: {result.get('t_statistic', np.nan):.4f}\n")
            para.add_run(f"p值: {result.get('p_value', np.nan):.6f}\n")
            para.add_run(f"F统计量: {result.get('f_statistic', np.nan):.4f}\n")
            para.add_run(f"F检验p值: {result.get('f_p_value', np.nan):.6f}\n")
            para.add_run(f"稳健方法: {result.get('variation', {}).get('robust_method', robust_method)}")
            
            metrics_y = result.get('metrics_y', {})
            para.add_run(f"\nY模型指标:\n")
            para.add_run(f"准确率: {metrics_y.get('accuracy', np.nan):.4f}\n")
            para.add_run(f"F1: {metrics_y.get('f1', np.nan):.4f}\n")
            para.add_run(f"AUC: {metrics_y.get('auc', np.nan):.4f}")
            
            metrics_t = result.get('metrics_t', {})
            para.add_run(f"\nT模型指标:\n")
            para.add_run(f"准确率: {metrics_t.get('accuracy', np.nan):.4f}\n")
            para.add_run(f"精确率: {metrics_t.get('precision', np.nan):.4f}\n")
            para.add_run(f"召回率: {metrics_t.get('recall', np.nan):.4f}\n")
            para.add_run(f"F1: {metrics_t.get('f1', np.nan):.4f}\n")
            para.add_run(f"AUC: {metrics_t.get('auc', np.nan):.4f}")
        else:
            para = doc.add_paragraph()
            para.add_run(f"场景 {name}: 错误 - {result['error']}")
    
    os.makedirs(save_path, exist_ok=True)
    
    word_file = os.path.join(save_path, f"analysis_results_{batch_id}_{timestamp_str}.docx")
    doc.save(word_file)
    print(f"详细结果已保存到Word文件: {word_file}")
    
    return word_file

def generate_word_report(group_results, save_path):
    """生成Word报告，汇报所有计算结果"""
    doc = Document()
    doc.add_heading('DML分析报告', 0)
    
    for group_key, results in group_results.items():
        doc.add_heading(f'分组: {group_key}', level=1)
        
        doc.add_heading('Y模型比较和选择', level=2)
        for model_name, metrics in results['y_metrics'].items():
            if metrics is not None:
                para = doc.add_paragraph(f"模型 {model_name}: ")
                para.add_run(f"准确率: {metrics.get('accuracy', 'N/A'):.4f}, ")
                para.add_run(f"精确率: {metrics.get('precision', 'N/A'):.4f}, ")
                para.add_run(f"召回率: {metrics.get('recall', 'N/A'):.4f}, ")
                para.add_run(f"F1: {metrics.get('f1', 'N/A'):.4f}, ")
                para.add_run(f"AUC: {metrics.get('auc', 'N/A'):.4f}")
        doc.add_paragraph(f"选择的Y模型: 集成模型 (CustomVotingClassifier)")
        
        doc.add_heading('T模型比较和选择', level=2)
        for model_name, metrics in results['t_metrics'].items():
            if metrics is not None:
                para = doc.add_paragraph(f"模型 {model_name}: ")
                para.add_run(f"准确率: {metrics.get('accuracy', 'N/A'):.4f}, ")
                para.add_run(f"精确率: {metrics.get('precision', 'N/A'):.4f}, ")
                para.add_run(f"召回率: {metrics.get('recall', 'N/A'):.4f}, ")
                para.add_run(f"F1: {metrics.get('f1', 'N/A'):.4f}, ")
                para.add_run(f"AUC: {metrics.get('auc', 'N/A'):.4f}")
        doc.add_paragraph(f"选择的T模型: 集成模型 (CustomVotingClassifier)")
        
        doc.add_heading('DML基准结果', level=2)
        dml_res = results['dml_results']
        para = doc.add_paragraph("处理效应估计: ")
        para.add_run(f"{dml_res.get('effect', 'N/A'):.6f}\n")
        para.add_run(f"标准误差: {dml_res.get('se', 'N/A'):.6f}\n")
        para.add_run(f"置信区间下界: {dml_res.get('lb', 'N/A'):.6f}\n")
        para.add_run(f"置信区间上界: {dml_res.get('ub', 'N/A'):.6f}\n")
        para.add_run(f"p值: {dml_res.get('p_value', 'N/A'):.6f}\n")
        para.add_run(f"t检验值: {dml_res.get('t_statistic', 'N/A'):.4f}\n")
        para.add_run(f"F检验值: {dml_res.get('f_statistic', 'N/A'):.4f}\n")
        para.add_run(f"f检验P值: {dml_res.get('f_p_value', 'N/A'):.6f}")
        
        doc.add_heading('敏感性分析结果', level=2)
        for sens_name, sens_res in results['sensitivity_results'].items():
            if 'error' not in sens_res:
                para = doc.add_paragraph(f"场景 {sens_name}: ")
                para.add_run(f"处理效应: {sens_res.get('effect', 'N/A'):.6f}, ")
                para.add_run(f"标准误差: {sens_res.get('se', 'N/A'):.6f}, ")
                para.add_run(f"置信区间下界: {sens_res.get('lb', 'N/A'):.6f}, ")
                para.add_run(f"置信区间上界: {sens_res.get('ub', 'N/A'):.6f}, ")
                para.add_run(f"t统计量: {sens_res.get('t_statistic', 'N/A'):.4f}, ")
                para.add_run(f"p值: {sens_res.get('p_value', 'N/A'):.6f}, ")
                para.add_run(f"F统计量: {sens_res.get('f_statistic', 'N/A'):.4f}, ")
                para.add_run(f"F检验p值: {sens_res.get('f_p_value', 'N/A'):.6f}")
                
                metrics_y = sens_res.get('metrics_y', {})
                para.add_run(f"\nY模型指标 - 准确率: {metrics_y.get('accuracy', 'N/A'):.4f}, 精确率: {metrics_y.get('precision', 'N/A'):.4f}, 召回率: {metrics_y.get('recall', 'N/A'):.4f}, F1: {metrics_y.get('f1', 'N/A'):.4f}, AUC: {metrics_y.get('auc', 'N/A'):.4f}")
                
                metrics_t = sens_res.get('metrics_t', {})
                para.add_run(f"\nT模型指标 - 准确率: {metrics_t.get('accuracy', 'N/A'):.4f}, 精确率: {metrics_t.get('precision', 'N/A'):.4f}, 召回率: {metrics_t.get('recall', 'N/A'):.4f}, F1: {metrics_t.get('f1', 'N/A'):.4f}, AUC: {metrics_t.get('auc', 'N/A'):.4f}")
            else:
                doc.add_paragraph(f"场景 {sens_name}: 错误 - {sens_res['error']}")
        
        doc.add_paragraph(results['sensitivity_summary'])
    
    word_file = os.path.join(save_path, f"dml_analysis_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.docx")
    doc.save(word_file)
    print(f"Word报告已生成: {word_file}")
    return word_file

if __name__ == "__main__":
    try:
        if torch.cuda.is_available():
            torch.cuda.set_per_process_memory_fraction(0.8)
            
        print("正在执行DML处理流程（集成学习版）")
        print(f"cuML版本: {cuml_version}")
        
        control_groups = {
            1: [
                'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
                'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize', 
                'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise', 
                'intelligence', 'person_status', 'party', 'workasso',
                'fid','pid','year'
            ],

            2: [
                'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
                'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize', 
                'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise', 
                'intelligence', 'person_status', 'party', 'workasso',
                'fid','pid','year',
                'ln_total_asset_sq', 'impro_sasti_sq', 'Child_sq',
                'lifefire_sq', 'age_sq', 'ln_renqing_sq', 'cfpsedu_sq',  'familysize_sq', 
                'Oldage_sq', 'health_sq', 'marriage_sq', 'ln_finc_sq', 'exercise_sq', 
                'intelligence_sq', 'person_status_sq'
            ],
            3: [
                'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
                'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize', 
                'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise', 
                'intelligence', 'person_status', 'party', 'workasso',
                'fid','pid','year',
                'ln_total_asset_sq', 'impro_sasti_sq', 'Child_sq',
                'lifefire_sq', 'age_sq', 'ln_renqing_sq', 'cfpsedu_sq',  'familysize_sq', 
                'Oldage_sq', 'health_sq', 'marriage_sq', 'ln_finc_sq', 'exercise_sq', 
                'intelligence_sq', 'person_status_sq', 
                'ln_total_asset_cub', 'impro_sasti_cub', 'Child_cub',
                'lifefire_cub', 'age_cub', 'ln_renqing_cub', 'cfpsedu_cub',  'familysize_cub', 
                'Oldage_cub', 'health_cub', 'marriage_cub', 'ln_finc_cub', 'exercise_cub', 
                'intelligence_cub', 'person_status_cub'
            ]
        }
        model_names = ['RandomForest', 'LogisticRegression', 'SVC', 'XGBoost', 'MLP']
        if HAS_GRADIENT_BOOSTING:
            model_names.append('GradientBoosting')
        
        save_path = "D:\\my-rapids-project\\result"
        os.makedirs(save_path, exist_ok=True)
        
        folds = [5]
        group_results = {}
        
        for fold in folds:
            for group_id, control_names in control_groups.items():
                group_key = f"{fold}fold_group{group_id}"
                print(f"\n=== 处理 {group_key} ===")
                
                preprocessor = DataPreprocessor(control_names)
                X, y, t, original_data = preprocessor.load_and_preprocess_data("l3dml_data_digital.dta")
                
                X_gpu = cp.array(X, dtype=cp.float32)
                y_gpu = cp.array(y, dtype=cp.float32)
                t_gpu = cp.array(t, dtype=cp.float32)
                
                y_comparator = ModelComparator('classification')
                t_comparator = ModelComparator('classification')
                
                print("\n=== 训练并比较Y模型（分类任务）===")
                y_models = {}
                y_metrics = {}
                for model_name in model_names:
                    try:
                        print(f"\n训练Y模型: {model_name}")
                        model, metrics = first_stage_cuml_enhanced(
                            X_gpu, y_gpu, 
                            task_type='classification', 
                            model_name=model_name
                        )
                        if model is not None and metrics is not None:
                            y_models[model_name] = model
                            y_metrics[model_name] = metrics
                            y_comparator.add_model(model_name, metrics)
                    except Exception as e:
                        print(f"训练Y模型 {model_name} 失败: {str(e)}")
                        traceback.print_exc()
                        y_metrics[model_name] = None
                
                print("\n=== 训练并比较T模型（分类任务）===")
                t_models = {}
                t_metrics = {}
                for model_name in model_names:
                    try:
                        print(f"\n训练T模型: {model_name}")
                        model, metrics = first_stage_cuml_enhanced(
                            X_gpu, t_gpu, 
                            task_type='classification', 
                            model_name=model_name
                        )
                        if model is not None and metrics is not None:
                            t_models[model_name] = model
                            t_metrics[model_name] = metrics
                            t_comparator.add_model(model_name, metrics)
                    except Exception as e:
                        print(f"训练T模型 {model_name} 失败: {str(e)}")
                        traceback.print_exc()
                        t_metrics[model_name] = None
                
                y_comparator.print_comparison()
                t_comparator.print_comparison()
                
                # 计算Y模型权重并创建集成模型原型（unfitted）
                model_factory = FirstStageModelFactory()
                y_scores = [y_comparator._calculate_score(y_metrics[name]) for name in model_names if y_metrics.get(name)]
                y_scores_array = np.array(y_scores)
                if len(y_scores_array) > 0:
                    max_score = np.max(y_scores_array)
                    y_weights = np.exp(y_scores_array - max_score) / np.sum(np.exp(y_scores_array - max_score))
                    y_weights = y_weights.tolist()  # Convert to list to avoid NumPy array issues
                else:
                    y_weights = []
                
                y_estimators = []
                input_size = X_gpu.shape[1]
                y_valid_model_names = [name for name in model_names if y_metrics.get(name)]
                for i, name in enumerate(y_valid_model_names):
                    if name == 'MLP':
                        model_proto = model_factory.get_model(name, 'classification', input_size=input_size)
                    else:
                        model_proto = model_factory.get_model(name, 'classification')
                    y_estimators.append((name, model_proto))
                
                ensemble_y_proto = CustomVotingClassifier(y_estimators, weights=y_weights) if y_estimators else None
                
                # 绘制Y模型集成权重图
                if y_weights and y_valid_model_names:
                    plot_ensemble_weights(
                        model_names=y_valid_model_names,
                        weights=y_weights,
                        save_path=save_path,
                        filename=f"y_ensemble_weights_{group_key}",
                        title=f"Y Ensemble Model Weights ({group_key})"
                    )
                
                # 计算T模型权重并创建集成模型原型（unfitted）
                t_scores = [t_comparator._calculate_score(t_metrics[name]) for name in model_names if t_metrics.get(name)]
                t_scores_array = np.array(t_scores)
                if len(t_scores_array) > 0:
                    max_score = np.max(t_scores_array)
                    t_weights = np.exp(t_scores_array - max_score) / np.sum(np.exp(t_scores_array - max_score))
                    t_weights = t_weights.tolist()  # Convert to list to avoid NumPy array issues
                else:
                    t_weights = []
                
                t_estimators = []
                t_valid_model_names = [name for name in model_names if t_metrics.get(name)]
                for i, name in enumerate(t_valid_model_names):
                    if name == 'MLP':
                        model_proto = model_factory.get_model(name, 'classification', input_size=input_size)
                    else:
                        model_proto = model_factory.get_model(name, 'classification')
                    t_estimators.append((name, model_proto))
                
                ensemble_t_proto = CustomVotingClassifier(t_estimators, weights=t_weights) if t_estimators else None
                
                # 绘制T模型集成权重图
                if t_weights and t_valid_model_names:
                    plot_ensemble_weights(
                        model_names=t_valid_model_names,
                        weights=t_weights,
                        save_path=save_path,
                        filename=f"t_ensemble_weights_{group_key}",
                        title=f"T Ensemble Model Weights ({group_key})"
                    )
                
                if ensemble_y_proto is None or ensemble_t_proto is None:
                    print(f"{group_key}: 未能成功训练任何模型，跳过DML分析")
                    continue
                
                print(f"\n{group_key} 选择的第一阶段Y集成模型: CustomVotingClassifier")
                print(f"\n{group_key} 选择的第一阶段T集成模型: CustomVotingClassifier")
                
                best_model_y = ensemble_y_proto
                best_model_t = ensemble_t_proto
                # 对于集成模型，metrics使用平均或代表性，这里用最佳单个的metrics作为代理
                best_y_model_name = y_comparator.get_best_model()
                best_t_model_name = t_comparator.get_best_model()
                best_metrics_y = y_comparator.model_metrics.get(best_y_model_name, {})
                best_metrics_t = t_comparator.model_metrics.get(best_t_model_name, {})
                
                print(f"\n{group_key} 步骤3: 执行完整的第二阶段DML分析")
                dml_results = torch_dml_with_pretrained(
                    X, y, t,
                    best_model_y,
                    best_model_t,
                    use_robust_se=True,
                    robust_method='bootstrap',
                    n_bootstrap=1000,
                    n_splits=fold
                )
                
                print(f"\n{group_key} 步骤4: 执行敏感性分析（优化版）")
                sensitivity_analyzer = SensitivityAnalysisOptimized(
                    X, y, t, control_names
                )
                
                variations = {
                    "基准模型": {
                        "control_vars": control_names,
                        "use_robust_se": True,
                        "robust_method": "bootstrap"
                    },
                    "简化控制变量": {
                        "control_vars": ['contracts', 'ln_total_asset', 'age', 'cfpsedu', 'familysize'],
                        "use_robust_se": True,
                        "robust_method": "bootstrap"
                    },
                    "不同稳健方法": {
                        "control_vars": control_names,
                        "use_robust_se": True,
                        "robust_method": "hc1"
                    }
                }
                
                sensitivity_results = sensitivity_analyzer.run_analysis(variations, n_splits=fold)
                
                try:
                    sensitivity_summary = sensitivity_analyzer.summarize()
                    print(sensitivity_summary)
                except AttributeError as e:
                    print(f"警告: 无法生成敏感性分析总结，方法可能未实现: {e}")
                    sensitivity_summary = "总结不可用"
                except Exception as e:
                    print(f"生成敏感性分析总结时出错: {e}")
                    sensitivity_summary = "总结生成出错"
                
                print(f"\n{group_key} 步骤5: 保存结果")
                save_detailed_results(
                    dml_results['effect'], dml_results['se'], dml_results['lb'], dml_results['ub'], 
                    dml_results['t_statistic'], dml_results['p_value'], dml_results['f_statistic'], dml_results['f_p_value'],
                    best_model_y, best_model_t, 
                    best_metrics_y, best_metrics_t,
                    sensitivity_results, 
                    save_path, 
                    robust_method='bootstrap',
                    batch_id=group_key
                )
                
                group_results[group_key] = {
                    'y_metrics': y_metrics,
                    't_metrics': t_metrics,
                    'best_y_model': 'CustomVotingClassifier',
                    'best_t_model': 'CustomVotingClassifier',
                    'dml_results': dml_results,
                    'sensitivity_results': sensitivity_results,
                    'sensitivity_summary': sensitivity_summary
                }
        
        generate_word_report(group_results, save_path)
        
        print("分析完成！结果已保存并生成Word报告。")
        
    except Exception as e:
        print(f"程序执行出错: {str(e)}")
        traceback.print_exc()
        with open(os.path.join(save_path, "error.log"), "a", encoding='utf-8') as log_file:
            log_file.write(f"[{datetime.now()}] 错误信息: {str(e)}\n")
        
    finally:
        print("执行终极资源清理...")
        clear_gpu_resources()
        
        if torch.cuda.is_available():
            print("最终GPU内存状态:")
            print(f"已用内存: {torch.cuda.memory_allocated()/1e9:.2f} GB")
            print(f"保留内存: {torch.cuda.memory_reserved()/1e9:.2f} GB")
            
        print("资源清理完成，程序退出！")  